# EV Penetration Presentation



In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repository root:", REPO_ROOT)

Repository root: c:\Users\qvdpm\Documents\GitHub\power-system-simulation-Sparky


## Overview

`run_ev_penetration` takes a base household load profile, selects houses on each feeder, adds EV charging profiles based on a seed, and gives the voltage and line performance.

## Inputs and Outputs

**Inputs**:
- `dataset`: power grid data including `sym_load` and network topology.
- `active_load_profiles`: time series of active power per house.
- `reactive_load_profiles`: matching reactive power per house.
- `ev_pool`: a set of EV charging profiles.
- `feeder_line_ids`: the top line IDs for each LV feeder.
- `graph_processor`: object with `find_downstream_vertices(feeder_line_id)`.
- `penetration_level`: fraction of houses with EVs (0.0 to 1.0).
- `random_seed`: used to make the assignment deterministic.

**Outputs**:
- `timestamp_table`: voltage summary per timestamp.
- `line_table`: summary per line, including losses and loading.

## Step 1: Validate inputs

The function first checks the `penetration_level`:
- must be a number
- must lie between `0.0` and `1.0`


## Step 2: Decide EV count per feeder

The number of EVs per feeder is calculated as:

`evs_per_feeder = floor(penetration_level * total_houses / number_of_feeders)`


## Step 3: Find downstream houses for each feeder

For each feeder line, the function uses `graph_processor.find_downstream_vertices(feeder_line_id)` to get all nodes on that feeder.

Then it maps those nodes to `sym_load` IDs so the EVs are assigned to actual houses.

## Step 4: Randomly select houses for EVs

Within each feeder:
- select `evs_per_feeder` houses from the downstream loads
- use the random seed for reproducibility
- use `replace=False` so a house is selected only once per feeder



## Step 5: Add EV charging to house profiles

For each selected house:
- pick an EV profile from `ev_pool`
- remove that profile from the available pool so it is not reused
- add the EV load to the house's active load profile

This creates an augmented active-load matrix containing both household and EV demand.

## Step 6: Run time-series power flow

Once the load profiles are updated, the function runs a time-series power flow.

The real implementation uses `PowerGridModel` to compute node voltages and line currents for every timestep.

## Step 7: Build summary tables

Two summary tables are created:

- `timestamp_table`: shows max/min voltage and the corresponding nodes per timestamp.
- `line_table`: shows total losses, max/min loading, and the timestamps when they occur.


In [6]:
from power_system_simulation.lv_grid_analytics import LVGridAnalytics

print("Starting EV penetration demo cell...")

TEST_DATA_PATH = "../tests/small_network"

try:
    grid = LVGridAnalytics(
        grid_path=f"{TEST_DATA_PATH}/input_network_data.json",
        feeder_line_ids=[16, 20],
        active_load_profile_path=f"{TEST_DATA_PATH}/active_power_profile.parquet",
        reactive_load_profile_path=f"{TEST_DATA_PATH}/reactive_power_profile.parquet",
        ev_profile_path=f"{TEST_DATA_PATH}/ev_active_power_profile.parquet",
    )

    timestamp_table, line_table = grid.run_ev_penetration(
        penetration_level=0.5,
        random_seed=42,
    )

    print("Timestamp table (voltage summary per timestamp):")
    print(timestamp_table)
    print("\n" + "=" * 60 + "\n")
    print("Line table (performance summary per line):")
    print(line_table)

except FileNotFoundError as e:
    print(f"File not found: {e}")
    print("Make sure to run this notebook from the 'example' directory.")
    print(f"Expected path: {TEST_DATA_PATH}")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Starting EV penetration demo cell...
Timestamp table (voltage summary per timestamp):
                     Max_Voltage  Max_Voltage_Node  Min_Voltage  \
Timestamp                                                         
2025-01-01 00:00:00     1.072931               1.0     1.049819   
2025-01-01 00:15:00     1.075911               1.0     1.050022   
2025-01-01 00:30:00     1.069725               1.0     1.049603   
2025-01-01 00:45:00     1.073244               1.0     1.049842   
2025-01-01 01:00:00     1.072924               1.0     1.049819   
...                          ...               ...          ...   
2025-01-10 22:45:00     1.071290               1.0     1.049723   
2025-01-10 23:00:00     1.075174               1.0     1.049986   
2025-01-10 23:15:00     1.072456               1.0     1.049796   
2025-01-10 23:30:00     1.071457               1.0     1.049734   
2025-01-10 23:45:00     1.076123               1.0     1.050049   

                     Min_Voltage_Node  
Ti

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_voltage = go.Figure()

fig_voltage.add_trace(
    go.Scatter(
        x=timestamp_table.index,
        y=timestamp_table["Max_Voltage"],
        mode="lines",
        name="Max Voltage",
        line={"color": "red", "width": 2},
    )
)

fig_voltage.add_trace(
    go.Scatter(
        x=timestamp_table.index,
        y=timestamp_table["Min_Voltage"],
        mode="lines",
        name="Min Voltage",
        line={"color": "blue", "width": 2},
        fill="tonexty",
        fillcolor="rgba(0,100,255,0.2)",
    )
)

fig_voltage.update_layout(
    title="Voltage Profile Over Time (EV Penetration: 50%)",
    xaxis_title="Timestamp",
    yaxis_title="Voltage (p.u.)",
    hovermode="x unified",
    template="plotly_white",
    height=500,
)

fig_voltage.show()

fig_lines = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=("Max Loading per Line", "Min Loading per Line", "Total Loss per Line"),
    specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]],
)

fig_lines.add_trace(
    go.Bar(x=line_table.index, y=line_table["Max_Loading"], name="Max Loading", marker_color="orange"), row=1, col=1
)

fig_lines.add_trace(
    go.Bar(x=line_table.index, y=line_table["Min_Loading"], name="Min Loading", marker_color="green"), row=2, col=1
)

fig_lines.add_trace(
    go.Bar(x=line_table.index, y=line_table["Total_Loss"], name="Total Loss", marker_color="red"), row=3, col=1
)

fig_lines.update_yaxes(title_text="Loading", row=1, col=1)
fig_lines.update_yaxes(title_text="Loading", row=2, col=1)
fig_lines.update_yaxes(title_text="Loss (kW)", row=3, col=1)
fig_lines.update_xaxes(title_text="Line ID", row=3, col=1)

fig_lines.update_layout(title_text="Line Performance Summary", height=900, showlegend=True, template="plotly_white")

fig_lines.show()